# Test-set evaluation of the movie recommender (Google Colab)

Evaluates a fine-tuned model (set `MODEL_NAME` in the setup cell: phase 2 or phase 3) on each user's **last** rating, which was never used
for training or validation:

- the model sees the user's previous 20 ratings and proposes movies with beam search that can only spell real
  catalogue titles; movies the user already rated are skipped
- **HR@10**: how often the real next movie is in the top 10; **NDCG@10**: also rewards ranking it higher
- baseline: recommend the most popular movies the user hasn't rated yet. A useful recommender has to beat it.

**Before you run:** Runtime → Change runtime type → GPU, then Runtime → Run all. Needs on Google Drive:
`MyDrive/GenRec/ml-1m/movies_wiki.csv`, `MyDrive/GenRec/ml-1m/ratings.dat` and the model in
`MyDrive/GenRec/<MODEL_NAME>/final`. Takes roughly 25 min for 1,000 users.

In [ ]:
import sys
already_loaded = "transformers" in sys.modules   # an old version in memory would mix with the new files

!nvidia-smi --query-gpu=name,memory.total --format=csv
# clean reinstall: replaces every file of whatever transformers version Colab ships with
%pip install -q --force-reinstall --no-deps transformers==5.18.0 accelerate==1.15.0
%pip install -q transformers==5.18.0 accelerate==1.15.0   # their dependencies, if missing

assert not already_loaded, "transformers was already imported: Runtime -> Restart session, then Run all again"

In [ ]:
import json
import math
import random
import re
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/GenRec")
DATA_DIR = DRIVE_DIR / "ml-1m"
MODEL_NAME = "qwen2.5-0.5b-movielens-rec"      # phase 2; "qwen2.5-0.5b-movielens-rec-v2" for phase 3
MODEL_DIR = DRIVE_DIR / MODEL_NAME / "final"
HISTORY_LEN = 20   # same as training (Config.history_len)

assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
for p in [DATA_DIR / "movies_wiki.csv", DATA_DIR / "ratings.dat", MODEL_DIR / "config.json"]:
    assert p.exists(), f"missing {p}"

## 1. Prompt format (identical to training)

In [ ]:
SYSTEM = "You are a movie expert who knows the MovieLens catalogue and recommends movies."
ARTICLES = r"The|A|An|La|Le|Les|L'|Il|El|Das|Der|Die"
ABBREVIATIONS = {"st", "dr", "mr", "mrs", "ms", "jr", "sr", "vs", "lt", "sgt", "capt", "col", "gen", "no"}
END = "<|im_end|>"

TEMPLATES = {
    "plot": ['What is the movie "{t}" about?', 'Summarize the plot of "{t}".', 'Tell me the story of the movie "{t}".'],
    "genres": ['Which genres does the movie "{t}" belong to?', 'What kind of movie is "{t}"?', 'List the genres of "{t}".'],
    "details": ['Who made the movie "{t}" and who stars in it?', 'Who directed "{t}" and who is in the cast?'],
    "identify": ["Which movie is this?\n\n{p}", "Name the movie with this plot:\n\n{p}"],
}


def display_title(ml_title):
    """'City of Lost Children, The (Cité des enfants perdus, La) (1995)' -> 'The City of Lost Children (1995)'"""
    m = re.match(r"^(.*?)\s*\((\d{4})\)\s*$", ml_title)
    name, year = (m.group(1), m.group(2)) if m else (ml_title, None)
    name = re.sub(r"\s*\([^()]*\)", "", name).strip() or name   # drop alternate titles
    name = re.sub(rf"^(.*), ({ARTICLES})$",
                  lambda a: a[2] + ("" if a[2].endswith("'") else " ") + a[1], name)
    return f"{name} ({year})" if year else name


def clean_plot(text):
    text = re.sub(r"\[\d+\]", "", str(text))   # citation markers
    return re.sub(r"\s+", " ", text).strip()


def truncate_words(text, n):
    """Cut to ~n words, ending on the last full sentence when that keeps at least half the text."""
    words = text.split()
    if len(words) <= n:
        return text
    cut = " ".join(words[:n])
    ends = [m.end() for m in re.finditer(r"[.!?][\"')]?(?=\s)", cut)
            if cut[:m.start()].split()[-1].lower().strip("\"'(") not in ABBREVIATIONS]
    return cut[:ends[-1]] if ends and ends[-1] > len(cut) * 0.5 else cut + " ..."


def mask_title(text, display):
    """Hide the movie's own name in an identify snippet so the answer isn't given away."""
    name = re.sub(r"\s*\(\d{4}\)$", "", display)
    for n in sorted({name, re.sub(r"^(The|A|An) ", "", name)}, key=len, reverse=True):
        if len(n) >= 4:   # skip very short names like "It" / "Big" that are ordinary words
            text = re.sub(rf"\b{re.escape(n)}\b", "___", text)
    return text


def is_known(value):
    return isinstance(value, str) and value.strip() and value.strip().lower() != "unknown"


def load_movies(data_dir):
    movies = pd.read_csv(data_dir / "movies_wiki.csv")
    movies["display"] = movies["Title"].map(display_title)
    # two different movies must never share a display title (it is the answer for identify / next_movie)
    dup = movies["display"].duplicated(keep=False)
    movies.loc[dup, "display"] = movies.loc[dup, "Title"]
    assert movies["display"].is_unique, movies.loc[movies["display"].duplicated(keep=False), "display"]
    return movies

In [ ]:
def next_movie_prompt(history, display, genres):
    lines = [f"{k}. {display[m]} - {genres[m]}" for k, m in enumerate(history, 1)]
    return ("Here are the movies a user rated, oldest first:\n" + "\n".join(lines)
            + "\n\nWhich movie will they rate next? Answer with the title and year.")

In [ ]:
def prompt_text(tokenizer, prompt):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## 2. Load the model

In [ ]:
device = "cuda"
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # L4/A100: bf16, T4: fp16
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=dtype).to(device).eval()
print(f"{MODEL_DIR.name}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params, {dtype}, "
      f"{torch.cuda.get_device_name()}")

## 3. Evaluate

In [ ]:
# Test-set evaluation: each user's LAST rating (never trained on, never used for validation).
# The model proposes TOP_K movies via beam search that can only spell real catalogue titles; movies the user already
# rated are skipped. HR@10 = target in the top 10; NDCG@10 also rewards ranking it higher. Baseline: most popular movies.
import math

from tqdm.auto import tqdm

EVAL_USERS = 1000   # random users to test - the same 1,000 as sasrec.ipynb and baselines.ipynb
TOP_K = 10
NUM_BEAMS = 30      # candidates per user before removing already-rated movies (keep 30: the reported numbers use it)
GEN_BATCH = 8       # users per generate() call

movies_df = load_movies(DATA_DIR)
display_of = dict(zip(movies_df.MovieID, movies_df.display))
genres_of = dict(zip(movies_df.MovieID, movies_df.Genres.str.replace("|", ", ")))
# "::"-separated; reading it as ":" with the fast C parser and keeping every other column is ~10x faster
ratings = pd.read_csv(DATA_DIR / "ratings.dat", sep=":", header=None, usecols=[0, 2, 4, 6],
                      names=["UserID", "MovieID", "Rating", "Timestamp"])
seqs = ratings.sort_values(["UserID", "Timestamp"], kind="stable").groupby("UserID").MovieID.apply(list)
users = random.Random(0).sample(list(seqs.index), min(EVAL_USERS, len(seqs)))

# token trie of every catalogue title (tokenized exactly like the training answers) -> beams can only produce real movies
end_id = tokenizer.convert_tokens_to_ids(END)
trie = {}
for title in movies_df.display:
    node = trie
    for tok in tokenizer(title + END, add_special_tokens=False).input_ids:
        node = node.setdefault(tok, {})


def ranked(scores, rated, k=TOP_K):
    """Top-k titles in order, skipping already-rated movies and duplicates."""
    out = []
    for title in scores:
        if title not in rated and title not in out:
            out.append(title)
        if len(out) == k:
            break
    return out


def hit_ndcg(top, target):
    if target not in top:
        return 0.0, 0.0
    return 1.0, 1 / math.log2(top.index(target) + 2)


@torch.no_grad()
def recommend(prompts):
    tokenizer.padding_side = "left"   # decoder-only batch generation: pad on the left
    texts = [prompt_text(tokenizer, p) for p in prompts]
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    tokenizer.padding_side = "right"
    start = enc.input_ids.shape[1]

    def allowed(batch_id, ids):
        node = trie
        for tok in ids[start:].tolist():
            node = node.get(tok)
            if node is None:
                return [end_id]
        return list(node) or [end_id]

    model.eval()
    out = model.generate(**enc, max_new_tokens=48, num_beams=NUM_BEAMS, num_return_sequences=NUM_BEAMS,
                         do_sample=False, use_cache=True, prefix_allowed_tokens_fn=allowed,
                         eos_token_id=end_id, pad_token_id=tokenizer.pad_token_id)
    titles = tokenizer.batch_decode(out[:, start:], skip_special_tokens=True)
    return [[t.strip() for t in titles[i * NUM_BEAMS:(i + 1) * NUM_BEAMS]] for i in range(len(prompts))]


# popularity baseline: rating counts without anyone's test item
popular = pd.Series([display_of[m] for items in seqs for m in items[:-1]]).value_counts().index.tolist()

prompt_of = {u: next_movie_prompt(seqs[u][:-1][-HISTORY_LEN:], display_of, genres_of) for u in users}
users.sort(key=lambda u: len(prompt_of[u]))   # similar lengths per batch: less padding

rows, t0 = [], time.time()
for b in tqdm(range(0, len(users), GEN_BATCH), desc="users", unit_scale=GEN_BATCH):
    batch_users = users[b:b + GEN_BATCH]
    for u, beams in zip(batch_users, recommend([prompt_of[u] for u in batch_users])):
        rated = {display_of[m] for m in seqs[u][:-1]}
        target = display_of[seqs[u][-1]]
        top = ranked(beams, rated)
        hit, ndcg = hit_ndcg(top, target)
        pop_hit, pop_ndcg = hit_ndcg(ranked(popular, rated), target)
        rows.append({"user": u, "target": target, "top10": top, "hit@1": float(top[:1] == [target]),
                     "hr@10": hit, "ndcg@10": ndcg, "pop_hr@10": pop_hit, "pop_ndcg@10": pop_ndcg})

results = pd.DataFrame(rows)
summary = pd.DataFrame({
    "model": [results["hit@1"].mean(), results["hr@10"].mean(), results["ndcg@10"].mean()],
    "most popular": [float("nan"), results["pop_hr@10"].mean(), results["pop_ndcg@10"].mean()],
}, index=["hit@1", "HR@10", "NDCG@10"])
print(f"{len(results)} test users, {time.time() - t0:.0f}s; avg {results.top10.map(len).mean():.1f} recommendations each "
      f"(below {TOP_K}: raise NUM_BEAMS)")
display(summary.style.format("{:.3f}"))

## 4. Look at some recommendations

In [ ]:
results.to_csv(DRIVE_DIR / f"eval_results_{MODEL_NAME}.csv", index=False)   # per-user top 10, kept on Drive
for _, r in results.sample(5, random_state=1).iterrows():
    history = [display_of[m] for m in seqs[r["user"]][:-1][-5:]]
    rank = r["top10"].index(r["target"]) + 1 if r["target"] in r["top10"] else None
    print(f"user {r['user']}  last rated: {'; '.join(history)}")
    print(f"   real next: {r['target']}   {f'HIT at rank {rank}' if rank else 'miss'}")
    print(f"   top 10:    {'; '.join(r['top10'])}\n")